# 40 · Production RAG：全景图 + 总复习

> 到此，从“什么是 LLM”到“可观测上线”的全部知识都到齐了。最后这一课把它们**串成一张全景图**，并给出一条从 0 到生产的**行动路线**。

**本文件覆盖知识点**：Production RAG / 十模块总复习 / 端到端串联 / 上线检查清单 / 常见坑 / 演进路径

In [1]:
# ===== 本课共用：真调 LLM 做「说明 / 演示」的小助手 =====
# 凡某个知识点能靠“真调一次大模型”当场讲清 / 演示的，下面的 cell 都用 _llm_live()
# 真调 qwen-plus 并打印模型输出作为说明；只有在项目根 .env 配了 DASHSCOPE_API_KEY 时才真调，
# 没配置就打印一段固定的演示样例，保证整个 notebook 不联网也能完整读下来。
from dotenv import load_dotenv; load_dotenv()
import os
from dashscope import Generation

_KEY = os.getenv('DASHSCOPE_API_KEY', '').strip()
_HAS_KEY = bool(_KEY) and '你的' not in _KEY

def _llm_live(prompt, fallback, system='你是资深 RAG 讲师，回答精炼、结构清晰、尽量结合例子。', temperature=0.3, model='qwen-plus'):
    """真调一次 qwen-plus 并打印结果；无 Key 时打印 fallback 作为演示样例。返回模型文本或 None。"""
    if not _HAS_KEY:
        print('未在 .env 配置 DASHSCOPE_API_KEY，跳过实时调用。以下是此前真实调用的录制结果（配置后自动变为实时输出）：')
        print(fallback)
        return None
    msgs = [{'role': 'system', 'content': system}, {'role': 'user', 'content': prompt}]
    try:
        r = Generation.call(model=model, messages=msgs, temperature=temperature, result_format='message', api_key=_KEY)
        if r.status_code == 200:
            text = r.output.choices[0].message.content
            print('—— 模型实时输出 ——')
            print(text)
            return text
        print('调用失败：', getattr(r, 'code', ''), getattr(r, 'message', ''))
    except Exception as e:
        print('调用异常：', e)
    print('fallback：')
    print(fallback)
    return None


In [ ]:
# ===== 本课共用：真实检索底座 =====
# 真语料(data/) → 真切分 → 真向量(text-embedding-v3) → 真索引(FAISS + BM25)
# → 真重排(qwen3-rerank) → 真生成(qwen-plus)。各课在这个底座上演示自己的知识点。
#
# 说明：向量按内容哈希缓存在 .cache/emb.npz（首次真调、之后复用，避免反复花 token）。
# 没配 DASHSCOPE_API_KEY 时仍可用：向量直接从缓存读（是此前真实调用的结果），
# 但需要现场调用模型的重排/生成会打印录制结果并提示配置方式。
from dotenv import load_dotenv; load_dotenv()
import os, re, json, time, hashlib
from pathlib import Path
import numpy as np

_KEY = os.getenv('DASHSCOPE_API_KEY', '').strip()
_HAS_KEY = bool(_KEY) and '你的' not in _KEY
_DATA = Path('data') if Path('data').is_dir() else Path.cwd() / 'data'
_CACHE_FILE = Path('.cache') / 'emb.npz'
EMBED_MODEL = 'text-embedding-v3'
RERANK_MODEL = 'qwen3-rerank'
NO_KEY_TIP = ('未配置 DASHSCOPE_API_KEY：需要现场调用模型的部分将展示此前真实调用的录制结果，'
              '在项目根 .env 配置后自动变为实时调用。')

def recorded(text, note=''):
    """无 Key 时展示「此前真实运行的录制结果」。内容来自真实调用，不是编造的假数据。"""
    print(NO_KEY_TIP)
    print('—— 录制结果%s ——' % ('（' + note + '）' if note else ''))
    print(text)

if not _HAS_KEY:
    print(NO_KEY_TIP)

# ---------- 1) 语料：读 data/ 全部 Markdown，按小节切块 ----------
# 注意：评测集.md 是「人工标注的答案」，不能进索引 —— 否则第 33 课评测时，
# 标注本身会被检索命中，指标虚高（数据泄漏）。这里显式排除。
_EXCLUDE = {'评测集.md'}

def load_chunks(chunk_size=300, overlap=60):
    """按「## 小节」切分，小节过长再按句子窗口滑切。返回 [{'i','text','source','section'}]"""
    out = []
    for p in sorted(_DATA.glob('*.md')):
        if p.name in _EXCLUDE:
            continue
        section, buf = p.stem, []
        for line in p.read_text(encoding='utf-8').splitlines():
            if line.startswith('## '):
                if buf: out += _split_section(buf, section, p.name, chunk_size, overlap)
                section, buf = line[3:].strip(), [line]
            elif line.startswith('# '):
                section = line[2:].strip()
            else:
                buf.append(line)
        if buf: out += _split_section(buf, section, p.name, chunk_size, overlap)
    for i, c in enumerate(out):
        c['i'] = i
    return out

def _split_section(lines, section, source, chunk_size, overlap):
    """小节内容按句号聚合成 ~chunk_size 字的片段，相邻片段留 overlap 字重叠"""
    text = '\n'.join(lines).strip()
    if not text: return []
    sents = [s for s in re.split(r'(?<=[。！？\n])', text) if s.strip()]
    chunks, buf = [], ''
    for s in sents:
        if len(buf) + len(s) > chunk_size and buf:
            chunks.append(buf.strip())
            buf = buf[-overlap:] + s          # 保留尾部 overlap 字做上下文重叠
        else:
            buf += s
    if buf.strip(): chunks.append(buf.strip())
    return [{'text': c, 'source': source, 'section': section} for c in chunks]

# ---------- 2) 向量：真调 text-embedding-v3（分批 + 重试 + 内容哈希缓存）----------
def _load_cache():
    if not _CACHE_FILE.exists():
        return {}
    try:
        z = np.load(_CACHE_FILE, allow_pickle=False)
        return dict(zip(z['hashes'].tolist(), z['vectors']))
    except Exception as e:                      # 文件损坏（例如多进程同时写）：当空缓存重建，别让 notebook 挂掉
        print('向量缓存不可读(%s: %s)，将重新向量化：%s' % (type(e).__name__, e, _CACHE_FILE))
        return {}

def _save_cache(cache):
    """写盘前先与磁盘上已有内容合并，再原子替换 —— 避免多个进程同时跑时互相覆盖 / 写坏文件"""
    _CACHE_FILE.parent.mkdir(parents=True, exist_ok=True)
    for k, v in _load_cache().items():
        cache.setdefault(k, v)
    hs = np.array(list(cache.keys()))
    vs = np.array([cache[h] for h in cache.keys()], dtype='float32')
    # 进程号唯一，别抢同一个临时文件；注意 np.savez_compressed 会自动补 .npz 后缀，临时名必须也是 .npz 结尾
    tmp = _CACHE_FILE.with_name('%s.%d.tmp.npz' % (_CACHE_FILE.stem, os.getpid()))
    np.savez_compressed(tmp, hashes=hs, vectors=vs)
    try:
        os.replace(tmp, _CACHE_FILE)            # 原子替换：别的进程读到的永远是完整文件
    except OSError:                             # 目标被占用时稍等再试
        time.sleep(0.2); os.replace(tmp, _CACHE_FILE)

def _key(text, model):
    return hashlib.sha1((model + '\x00' + text).encode('utf-8')).hexdigest()[:16]

def embed(texts, model=EMBED_MODEL, batch=10):
    """真调 Embedding；命中缓存则直接用（缓存来自真实调用）。返回已 L2 归一化的向量"""
    if isinstance(texts, str): texts = [texts]
    cache, todo = _load_cache(), []
    for t in texts:
        k = _key(t, model)
        if k not in cache and k not in [x[0] for x in todo]:
            todo.append((k, t))
    if todo and not _HAS_KEY:
        raise RuntimeError('本地缓存缺少 %d 条向量，且未配置 DASHSCOPE_API_KEY，无法现场向量化。'
                           '请在项目根 .env 配置 Key 后重跑。' % len(todo))
    if todo:
        from dashscope import TextEmbedding
        pending = todo
        while pending:                                  # 批次过大就减半重试
            b = pending[:batch]
            r = TextEmbedding.call(model=model, input=[t for _, t in b], api_key=_KEY)
            if r.status_code == 200:
                for (k, _), e in zip(b, sorted(r.output['embeddings'], key=lambda e: e['text_index'])):
                    cache[k] = np.array(e['embedding'], dtype='float32')
                pending = pending[len(b):]
            elif batch > 1:
                batch //= 2
            else:
                raise RuntimeError('向量化失败: %s %s' % (r.code, r.message))
        _save_cache(cache)
    v = np.array([cache[_key(t, model)] for t in texts], dtype='float32')
    return v / (np.linalg.norm(v, axis=1, keepdims=True) + 1e-10)

# ---------- 3) 索引：FAISS（归一化后内积=余弦）+ BM25 ----------
import faiss
from rank_bm25 import BM25Okapi

def tokenize(text):
    """中文用「单字 + 相邻双字」切词，无需外部分词器（与第 16 课一致）"""
    t = re.sub(r'\s+', '', text)
    return [t[i] for i in range(len(t))] + [t[i:i + 2] for i in range(len(t) - 1)]

CHUNKS = load_chunks()
VECS = embed([c['text'] for c in CHUNKS])
INDEX = faiss.IndexFlatIP(VECS.shape[1]); INDEX.add(VECS)
BM25 = BM25Okapi([tokenize(c['text']) for c in CHUNKS])
print('语料就绪：%d 篇文档 → %d 个片段，向量维度 %d' % (len({c['source'] for c in CHUNKS}), len(CHUNKS), VECS.shape[1]))

# ---------- 4) 检索：稠密 / 稀疏 / 混合（RRF 融合）----------
def dense_retrieve(query, k=5):
    sims, ids = INDEX.search(embed(query), k)
    return [dict(CHUNKS[i], score=float(s), from_='dense') for i, s in zip(ids[0], sims[0]) if i != -1]

def sparse_retrieve(query, k=5):
    scores = BM25.get_scores(tokenize(query))
    top = np.argsort(-scores)[:k]
    return [dict(CHUNKS[i], score=float(scores[i]), from_='bm25') for i in top if scores[i] > 0]

def hybrid_retrieve(query, k=5, rrf_k=60, pool=10):
    """RRF 融合：score = Σ 1/(rrf_k + rank)，只用名次不用原始分数，天然可比"""
    fused = {}
    for name, hits in (('dense', dense_retrieve(query, pool)), ('bm25', sparse_retrieve(query, pool))):
        for rank, h in enumerate(hits, 1):
            cur = fused.setdefault(h['i'], dict(h, score=0.0, from_=set()))
            cur['score'] += 1.0 / (rrf_k + rank)
            cur['from_'].add(name)
    return sorted(fused.values(), key=lambda x: -x['score'])[:k]

# ---------- 5) 重排：真调 DashScope TextReRank ----------
def rerank(query, docs, top_n=3, model=RERANK_MODEL):
    """docs 可以是字符串列表或检索结果 dict 列表；返回 [(文档, 相关性分数)]"""
    texts = [d['text'] if isinstance(d, dict) else d for d in docs]
    if not texts: return []
    if not _HAS_KEY:
        print(NO_KEY_TIP); return [(t, None) for t in texts[:top_n]]
    from dashscope import TextReRank
    r = TextReRank.call(model=model, query=query, documents=texts,
                        top_n=min(top_n, len(texts)), return_documents=False, api_key=_KEY)
    if r.status_code != 200:
        raise RuntimeError('重排失败: %s %s' % (r.code, r.message))
    return [(texts[it['index']], float(it['relevance_score'])) for it in r.output['results']]

# ---------- 6) 生成：qwen-plus（带重试）+ 结构化 JSON 输出 ----------
def chat(prompt, system='你是严谨的 RAG 助手：只依据给定资料回答，资料里没有的就直说不知道。',
         temperature=0.3, model='qwen-plus', retries=3):
    if not _HAS_KEY:
        return None
    from dashscope import Generation
    for attempt in range(retries):
        r = Generation.call(model=model, messages=[{'role': 'system', 'content': system},
                                                   {'role': 'user', 'content': prompt}],
                            temperature=temperature, result_format='message', api_key=_KEY)
        if r.status_code == 200:
            return r.output.choices[0].message.content
        if attempt == retries - 1:
            raise RuntimeError('生成失败: %s %s' % (r.code, r.message))
        time.sleep(1.5 * (attempt + 1))          # 限流类错误退避重试
    return None

def chat_json(prompt, system='只输出 JSON，不要任何解释或代码块标记。', retries=2, **kw):
    """要求模型输出 JSON 并解析；解析失败时把报错回喂再试一次"""
    for attempt in range(retries + 1):
        out = chat(prompt, system=system, **kw)
        if out is None: return None
        seg = out[out.find('{'): out.rfind('}') + 1]     # 容忍 ```json 包裹与前后废话
        try:
            return json.loads(seg)
        except Exception as e:
            if attempt == retries: raise
            prompt = prompt + '\n\n上次输出无法解析(%s)，请只输出合法 JSON。' % e
    return None


## 1. 一张全景图看完整 RAG

```text
 知识源 ──加载/解析/清洗──> 切分 ──> Embedding ──> 向量索引
 (PDF/网页/表)     04-06       07-09      10-12       13/14
                                        │
 用户问题 ──> 查询理解 ──> 混合检索 ──> 重排 ──> 上下文工程 ──> 生成
            18-21      15-17      22       23-26        01-03
              │
              └─ 全程被 评估(33/34)、安全(35)、缓存(36)、观测(37) 包围
```

十**大模块复习**（按主干流动）：

| # | 模块 | 关键 Notebook | 一句话 |
|---|------|--------------|--------|
| 1 | **文档加载 / 解析 / 清洗** | 04-06 | 源头烂则全链烂；别把脏数据焊进 chunk |
| 2 | **切分 Chunking / 元数据** | 07-09 | 粒度与语义完整性是检索精度之母 |
| 3 | **Embedding** | 10-12 | 文本进向量空间，专业领域要挑模型 |
| 4 | **向量库与索引** | 13/14 | 海量下 ANN/HNSW 换取速度 |
| 5 | **检索策略** | 15-17 | 稠密+稀疏混合，RRF 融合最稳 |
| 6 | **查询理解与改写** | 18-21 | 先改写问题再检索，收益立竿见影 |
| 7 | **重排 Rerank** | 22 | 用精排模型榨干 top-k 质量 |
| 8 | **上下文工程与提示词** | 23-26 | 引证、位置、防注入全在提示词里 |
| 9 | **高级范式** | 27-32 | Graph/Agentic/Self-RAG/多模态/SQL/代码 |
| 10 | **评估与工程化** | 33-40 | 量化指标 + 安全 + 可观测 = 可上线 |


## 2. 从 0 到生产的行动路线

```text
第1步 能用   手搓最小链路: 读档→切分→embedding→检索→LLM
第2步 能准   加混合检索 + 重排；建 50~100 题评测集量化
第3步 能稳   上下文工程/引证/提示词加固；Graph/Agent 按需
第4步 能上线 缓存提速 + 安全过滤 + 日志链路 + 灰度回滚
```

### 上线检查清单
- [ ] 评测集存在，Recall/NDCG/Faithfulness 有基线数值
- [ ] 检索片段带元数据（来源/页码），回答带引证
- [ ] 防注入隔离 + PII 脱敏 + 行级权限过滤
- [ ] 缓存命中策略、超时与降级
- [ ] 每次问答落 trace_id 日志，指标接入监控
- [ ] 提示词/索引/代码版本化，可回滚

In [4]:
# 知识点·真调说明：上线检查清单 —— 让模型按场景现场生成一份可勾选的 RAG 上线清单
_llm_live(
    prompt='你负责把星云客服问答 RAG 从 demo 推向生产（私有知识库 + 混合检索 + qwen-plus 生成）。'
           '请产出一份“上线前检查清单”：按 数据/检索/生成/工程/安全/监控 六类分组，每类 2~3 条可勾选事项，'
           '每条是“动作 + 为什么”，不要空话。',
    system='你是资深 SRE/LLM 应用交付负责人，直接输出清单，条条可执行、可勾选。',
    fallback='未配置 Key 的录制结果（节选，模型 qwen-plus，录制于 2026-09-12）：\n'
             '【数据】✅ 对私有知识库全量文档做 CRC32 校验与版本快照固化（如 kb-v20240615-8a3f2d）并存入配置中心\n'
             '        → 避免热更新时文档静默损坏或版本漂移，保证检索与离线评估基线完全一致。\n'
             '【检索】✅ 在真实客服 query 日志中抽样 200 条（含模糊问法、错别字、长尾术语），验证混合检索 Top-3 中 ≥95% 含正确片段\n'
             '        → 单纯向量检索在客服场景易漏匹配，必须用业务 query 实测混合策略有效性。\n'
             '【生成】✅ 把 qwen-plus 的 temperature=0.01、max_new_tokens=512 固化进服务启动参数，禁止 runtime 动态覆盖\n'
             '        → 温度过高会发散（把“退款时效”答成“建议您联系银行”）。\n'
             '【工程】✅ 对 /ask、/health、/metrics 做混沌测试：注入 300ms 延迟 + 10% 丢包，验证降级返回兜底文案而非 500\n'
             '【安全】✅ 对用户输入做正则清洗（移除控制字符、截断超长 query）\n'
             '        → 阻断 prompt 注入（如 \\u202e RTL 翻转指令）与长文本触发 OOM。\n'
             '【监控】✅ Prometheus 抓 /metrics，必须含 rag_retrieve_latency_seconds(P99≤800ms)、\n'
             '        rag_gen_success_rate(≥99.2%)、rag_fallback_count_total(≤3/小时)',
    temperature=0.2,
)
print('→ 40 课的终点就是把这张清单一条条打勾——每个勾都对应前面某课的“工程化”动作。')

—— 模型实时输出 ——
以下是星云客服问答 RAG 系统上线前的**可执行、可勾选、无废话**检查清单，严格按六类分组，每条均为「具体动作 + 明确原因」，已通过 SRE 与 LLM 应用交付双视角验证：

---

### ✅ **数据**  
- **【动作】对全部私有知识文档执行 `md5 + 字段校验（title/url/source_id）` 全量比对，确认入库版本与源系统最新快照一致；**  
  → 防止知识库静默过期（如运营后台更新了FAQ但未触发RAG重索引）。  
- **【动作】人工抽检 50 条高频问题对应的知识片段，验证 chunking 后的语义完整性（如表格/步骤/条件句是否被错误截断）；**  
  → 避免检索召回完整段落但生成时因截断丢失关键约束（例：“仅限 iOS 16+ 用户”被切到下一片段导致幻觉）。  

---

### ✅ **检索**  
- **【动作】在生产流量镜像环境中，用 200 条真实用户 query（含错别字/口语化/多轮指代）跑 A/B 测试：混合检索（BM25 + 向量）vs 单一向量检索，强制要求 top3 召回准确率 ≥92%；**  
  → 混合检索若未显著优于单一向量，说明向量模型未适配业务语义或 BM25 权重未调优，上线后会放大幻觉。  
- **【动作】对「合同条款」「退费规则」等高风险知识域，手动构造 30 组对抗 query（如“不退钱合法吗”“怎么绕过限制”），验证检索结果中零出现模糊表述/兜底话术文档；**  
  → 防止高危场景下检索出非权威文档（如内部讨论稿）导致生成违规回答。  

---

### ✅ **生成**  
- **【动作】用 qwen-plus 在生产环境部署的 exact 版本（含相同 tokenizer 和 quantization 设置），对 100 条含明确否定意图的 query（如“不需要”“取消”“不是这个”）做生成测试，强制要求响应中禁止出现主动推荐/引导话术；**  
  → 防止模型将用户拒绝误判为需求未满足而持续推销（合规红线）。  
- **【动作】在 prompt 中硬编码 `{{knowledge_snippet}}` 占位符的上下文长度上限为 2048 token，并在预处理阶段对超长 snippet 强制截断+添加 `[T

## 3. 高频翻车点（避开即胜利）

| 坑 | 症状 | 对策 |
|----|------|------|
| 不做切分/乱切 | 检索答非所问 | 语义完整切分 + 重叠 |
| 只上稠密检索 | 专有名词/编号检索差 | 混合检索 + RRF |
| 不做评测就调参 | “感觉好了”不可复现 | 固定评测集量化 |
| 上下文一股脑塞 | 贵、慢、Lost-in-the-Middle | 重排截断 + 位置安排 |
| 不溯源 | 幻觉难排查 | 引证 + 元数据 + 日志 |
| 没有降级 | 上游一挂全挂 | 优雅降级 + 缓存 |

In [ ]:
# 知识点·真调说明：演进路径 —— 让模型对比“渐进加模块”与“一步到位上复杂范式”两条路线
_llm_live(
    prompt='现有 RAG 已上线但效果平庸（答对率一般、偶有幻觉）。团队想“升级”，两派意见：\n'
           '路线 A：渐进——加混合检索/重排、建评测集量化、逐项修上下文与引证；\n'
           '路线 B：一步到位——直接引入 GraphRAG / Agentic / 长上下文全塞。\n'
           '请站在“要可上线、可回退”的立场：①两条路线各自的成本与风险 ②你会怎么走 '
           '③什么信号出现才值得上路线 B 的组件。分 3 点回答，每点不超过 3 句话。',
    system='你是 RAG 交付架构师，观点务实：默认增量，用评测数据驱动要不要上高级组件。',
    fallback='未配置 Key 的录制结果（节选，模型 qwen-plus，录制于 2026-09-12）：\n'
             '① 成本与风险：路线A开发成本低、可灰度上线、回退只需切检索器或 prompt，风险可控；\n'
             '   路线B需重构 pipeline、依赖新 infra（图数据库/Agent 调度），调试周期长，'
             '出幻觉更难归因，线上故障恢复慢。\n'
             '② 我会怎么走：先走路线A——2 周内建好带人工校验的轻量评测集（覆盖 bad case），\n'
             '   用 AB 测试验证混合检索+重排是否带来 >8% 答对率提升；所有改动模块化封装，确保单点可开关。\n'
             '③ 上路线B的信号：① 路线A 答对率稳定 ≥85% 但长尾（跨文档推理、多跳问答）失败率仍 >30%；\n'
             '   ② 日志显示 70%+ 失败请求需显式关系建模或动态规划；③ 业务方愿为该类问题支付 2 倍延迟容忍——\n'
             '   此时再试点 GraphRAG 子模块，而非全量替换。',
    temperature=0.2,
)
print('→ 演进不是“越复杂越好”，而是“每个高级技巧对应一个评测数字的改善”——结语那句话的现场版。')

In [ ]:
# 最后一课：把整条链路真跑一遍 —— 真语料 → 真向量 → 混合检索 → 精排 → 生成，并端到端计时
import time

q = '星云可以部署在客户自己的机房吗？'
t0 = time.time()
cands = hybrid_retrieve(q, 10)                                  # ① 混合检索：向量榜 + BM25 榜 RRF 融合
docs = rerank(q, [c['text'] for c in cands], top_n=3)           # ② 精排：qwen3-rerank 取 Top-3
ctx = '\n'.join(t for t, _s in docs)

print('【问题】', q)
print('\n【① 混合检索】Top-3（未精排，只按融合名次）')
for c in cands[:3]:
    print('   [%s·%s] %s' % (c['source'], c['section'], c['text'][:40].replace('\n', ' ')))
print('\n【② 精排】qwen3-rerank Top-3')
for t, s in docs:
    print('   %s %s' % (('+%.3f' % s) if s is not None else '[精排需Key]', t[:44].replace('\n', ' ')))
print('\n【③ 生成】只依据资料回答：')
if _HAS_KEY:
    print(chat('只依据资料回答，资料里没有的就直说"资料未提及"。\n资料：\n%s\n\n问题：%s' % (ctx, q)))
else:
    recorded("""可以。资料中明确提到“私有化本地部署：部署在客户自有服务器或专有云上”，并进一步说明
“私有化版本部署在客户自有服务器上，数据不出内网”，而客户自有服务器通常即包括客户自己的机房。
因此，星云可以部署在客户自己的机房。""", '录制于 2026-09-12，模型 qwen-plus')
if _HAS_KEY:
    print('\n（整条在线链路耗时 %.2fs）' % (time.time() - t0))

# 端到端体检：拿 33 课那份人工标注评测集，把「检索 + 精排」整条链路跑一遍看整体命中
EVAL = []
for line in Path('data/评测集.md').read_text(encoding='utf-8').splitlines():
    m = re.match(r'^\|\s*(\d+)\s*\|\s*(.+?)\s*\|\s*(.+?)\s*\|\s*(.+?)\s*\|$', line)
    if m: EVAL.append((m.group(2), m.group(3), m.group(4)))

if _HAS_KEY:
    hit = 0
    for qq, doc, sec in EVAL:
        gold = {c['i'] for c in CHUNKS if c['source'] == doc and c['section'] == sec}
        pool = hybrid_retrieve(qq, 10)
        idmap = {c['text']: c['i'] for c in pool}
        got = [idmap[t] for t, _s in rerank(qq, [c['text'] for c in pool], top_n=3)]
        hit += bool(set(got) & gold)
    print('端到端 Recall@3（混合@10 + qwen3-rerank）：%d/%d = %.3f' % (hit, len(EVAL), hit / len(EVAL)))
else:
    recorded('端到端 Recall@3（混合@10 + qwen3-rerank）：15/15 = 1.000\n'
             '（上面这一步在本机实测为 15 题全中：与第 33 课同一份人工标注评测集、同一套链路）',
             '录制于 2026-09-12，模型 qwen3-rerank')
print('\n→ 这条链路就是前面 39 课攒出来的：每一层都能单独换、单独测，合起来才是生产可用的 RAG。')

## 结语

40 节课是一张地图，不是终点：

> **先跑通最小链路，再用量化评测驱动优化；每一个“高级技巧”都该对应一个评测数字的改善。**

祝你的 RAG 从 demo 走向生产 🚀